<a href="https://colab.research.google.com/github/AhmedE16/flyrank-ai-intern-ahmed-essam/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    LIMIT 1
""").df()
print(schema)

print("""
CONTRACT

1) Unit of analysis: one row = one page, on one day (content_hash_id x report_date),
   from fact_content_daily_performance.

2) Table(s): fact_content_daily_performance (daily signals). dim_content and
   dim_clients not joined this week -- staying at the daily-fact grain only, to
   keep the contract small and verifiable.

3) Time window: a mid-panel month, month=2026-03 -- NOT the _sample file, which is
   the sealed final month and must stay untouched until final evaluation.

4) Label / proxy: is_declining, built from within-month movement -- a page whose
   second-half impressions drop meaningfully below its first-half impressions.
   This is a same-window proxy, not a true future outcome; a stronger version
   (prior month -> next month decline) is a Week 5+ task once I build multi-month
   windows.

5) One deliberate exclusion: fact_content_query_90d (query-level table) -- not
   touched this week. My lane operates at the page level, not the query level, and
   mixing grains before locking page-level features risks leakage through
   query-level aggregates that already encode outcome information.
""")

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
print("""
FIELD BUCKETS

FEATURES (knowable at decision time):
- gsc_clicks          -- observed search signal, known same-day
- gsc_avg_position     -- reported by Search Console, known same-day
- ga4_sessions          -- observed analytics signal, known same-day
- gsc_impressions (first half of month only) -- known well before month-end

LABEL / PROXY:
- is_declining -- derived by comparing first-half vs second-half monthly
  impressions. This IS the target, never a feature.

CONTEXT (background, not features, not the label):
- client_hash_id, content_hash_id  -- join keys only, carry no signal themselves
- gsc_data_available, ga4_data_available -- used only to check coverage, not fed
  into the model as predictive signal

EXCLUDED:
- fact_content_query_90d           -- wrong grain (query-level), excluded this week
- gsc_impressions (second half of month) -- this is what the label is computed
  from; excluded from features on purpose (see the leakage trap below)
- any product-computed score (health_score, priority_score, action_type) -- not
  shipped in this data by design, standing rule: never rebuild one and feed it
  back in as a feature.
""")


FIELD BUCKETS

FEATURES (knowable at decision time):
- gsc_clicks          -- observed search signal, known same-day
- gsc_avg_position     -- reported by Search Console, known same-day
- ga4_sessions          -- observed analytics signal, known same-day
- gsc_impressions (first half of month only) -- known well before month-end

LABEL / PROXY:
- is_declining -- derived by comparing first-half vs second-half monthly
  impressions. This IS the target, never a feature.

CONTEXT (background, not features, not the label):
- client_hash_id, content_hash_id  -- join keys only, carry no signal themselves
- gsc_data_available, ga4_data_available -- used only to check coverage, not fed
  into the model as predictive signal

EXCLUDED:
- fact_content_query_90d           -- wrong grain (query-level), excluded this week
- gsc_impressions (second half of month) -- this is what the label is computed
  from; excluded from features on purpose (see the leakage trap below)
- any product-computed score (

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# QUERY 1 -- grain check
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (content_hash_id, report_date) pairs found: {len(grain_check)}")
print("Zero above confirms the grain: one row = one page, one day.\n")

# QUERY 2 -- row count + date span
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_pages
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").df()
print(span, "\n")

# QUERY 3 -- availability, filtered with IS TRUE
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").df()
print(avail)
print(f"{avail['ga4_available_rows'][0]:,} of {avail['total_rows'][0]:,} rows have GA4 data available this month.\n")

# Build a page-level feature frame for month=2026-03
page_month = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions_month,
        SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_second_half
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY content_hash_id
""").df()

page_month["is_declining"] = (
    (page_month["impressions_first_half"] > 0) &
    (page_month["impressions_second_half"] < page_month["impressions_first_half"] * 0.8)
).astype(int)

print(page_month.head(10))
print(f"\nDeclining share this month: {page_month['is_declining'].mean():.1%}\n")

print("""
FEATURES -- available at decision moment?
1. clicks_month             -- available when: Search Console reports it same-day, known before any decision
2. avg_position               -- available when: reported same-day by Search Console, no future info used
3. sessions_month              -- available when: Analytics reports it same-day, known before any decision
4. impressions_first_half        -- available when: only uses the first 15 days, known well before month-end
5. content_hash_id (join/presence) -- available when: a join key, always known
""")

# THE TRAP -- deliberate leakage, then remove it
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X_honest = page_month[["clicks_month", "avg_position", "sessions_month", "impressions_first_half"]].fillna(0)
y = page_month["is_declining"]

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_honest, y)
honest_auc = roc_auc_score(y, tree.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (features only): {honest_auc:.3f}")

# Deliberately leak: add decline_ratio, which is almost exactly what the label
# threshold (< 0.8) was computed from -- this is the "answer key" leak
X_leaky = X_honest.copy()
X_leaky["decline_ratio"] = (
    page_month["impressions_second_half"] / page_month["impressions_first_half"].replace(0, 1)
)

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_auc = roc_auc_score(y, tree_leaky.predict_proba(X_leaky)[:, 1])
print(f"'Leaky' AUC (with decline_ratio added): {leaky_auc:.3f}")

print(f"\nJump from {honest_auc:.3f} to {leaky_auc:.3f} is not real improvement --")
print("decline_ratio is almost exactly the quantity the is_declining threshold (< 0.8) was computed from,")
print("so the model is just reading the answer key back, not learning a pattern.")
print("Deleting decline_ratio and keeping the honest AUC as the real result.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content_hash_id, report_date) pairs found: 0
Zero above confirms the grain: one row = one page, one day.

    n_rows   min_date   max_date  n_pages
0  9841378 2026-03-01 2026-03-31   331437 



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378              413966
413,966 of 9,841,378 rows have GA4 data available this month.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  impressions_month  clicks_month  avg_position  \
0  content_b7e512995f79d5a6             1140.0           2.0      4.394234   
1  content_05597932fe4da067               57.0           0.0      2.714744   
2  content_905aa32a0230694e              149.0           0.0      6.481453   
3  content_05434271b257bb68             1421.0           6.0      6.320337   
4  content_d056587ff7faca0c             2770.0          16.0      4.459107   
5  content_bfd1e41c2af250c8               48.0           0.0     14.753175   
6  content_2662845f598544ef              150.0           1.0      6.341880   
7  content_22610b0934f8825e               67.0           0.0     12.791667   
8  content_712c365258cee05c             6048.0          23.0      4.950311   
9  content_476c37c366920c1b              223.0           0.0     50.390299   

   sessions_month  impressions_first_half  impressions_second_half  \
0             0.0                   429.0                    711.0   
1

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
print("""
LIMITATION: this is an unbalanced panel -- per-client history depth differs, and
some rows carry gsc_data_available = FALSE or ga4_data_available = FALSE when a
client's tracking hadn't started yet. This month=2026-03 slice also can't tell me
anything about longer-term trend or seasonality -- it's a single 31-day window, so
a page that looks "declining" here might just be mid-way through a normal monthly
fluctuation rather than a real sustained decline. That distinction needs a
multi-month window, which is future work.
""")


LIMITATION: this is an unbalanced panel -- per-client history depth differs, and
some rows carry gsc_data_available = FALSE or ga4_data_available = FALSE when a
client's tracking hadn't started yet. This month=2026-03 slice also can't tell me
anything about longer-term trend or seasonality -- it's a single 31-day window, so
a page that looks "declining" here might just be mid-way through a normal monthly
fluctuation rather than a real sustained decline. That distinction needs a
multi-month window, which is future work.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.